# Lung Cancer Classification: Improved Hybrid Multi-Model Architecture

This notebook presents a significantly enhanced hybrid ensemble for lung cancer classification, designed with a focus on leveraging pre-trained weights for robust feature extraction and incorporating specialized components for micro-nodule detection. The architecture integrates Vision Transformers (ViT), DenseNet, EfficientNet, and a novel Custom CNN, combined through an adaptive fusion module.

This updated version reflects critical improvements identified during review, ensuring a more effective and scientifically sound approach to medical image classification.

### Imports and Environment Setup

This section installs necessary libraries and sets up the computational environment, including device (GPU/CPU) detection and essential imports.

In [1]:
# =========================
# IMPORTS & ENVIRONMENT SETUP
# =========================
!pip install -q timm scikit-learn seaborn matplotlib kagglehub

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
import timm
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
import os
import shutil
import kagglehub
from google.colab import files
from PIL import Image
import io

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] Using Device:", device)

# Assuming dataset is already downloaded as per user's prompt
# The variables download_path and base_dir should be available from previous execution.

[INFO] Using Device: cpu


In [2]:
import kagglehub
import os

# CELL 2: Download Dataset
print("[INFO] Downloading dataset...")
dataset_handle = "mohamedhanyyy/chest-ctscan-images"
download_path = kagglehub.dataset_download(dataset_handle)
print(f"[INFO] Downloaded to: {download_path}")

# Explicitly set base_dir to the 'Data' subdirectory
# This bypasses the potentially problematic os.walk logic and ensures consistency
# with the dataset structure where 'train', 'valid', 'test' are under 'Data'.
base_dir = os.path.join(download_path, 'Data')

# Verify the base_dir and its contents
if not os.path.exists(base_dir):
    print(f"[ERROR] Expected data directory 'Data' not found under {download_path}. Please check dataset structure.")
    # Fallback to the download_path itself if 'Data' isn't found, though this might lead to further errors
    base_dir = download_path # This fallback might still be problematic if structure varies
    print(f"[INFO] Falling back to download_path as base_dir: {base_dir}")

print(f"[INFO] Data directory: {base_dir}")
print(f"[INFO] Classes: {[d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]}")

[INFO] Downloading dataset...
Using Colab cache for faster access to the 'chest-ctscan-images' dataset.
[INFO] Downloaded to: /kaggle/input/chest-ctscan-images
[INFO] Data directory: /kaggle/input/chest-ctscan-images/Data
[INFO] Classes: ['valid', 'test', 'train']


### Data Augmentation and Pipeline

This section defines the image transformations for training and testing, and sets up `DataLoader` objects to efficiently load data in batches. A validation split is also introduced for robust model monitoring.

In [3]:
import os # Added import
import torch # Added import
from torchvision import datasets, transforms # Added import

# =========================
# DATA AUGMENTATION & PIPELINE
# =========================
# `base_dir` is already correctly defined from the previous cell as:
# /root/.cache/kagglehub/datasets/mohamedhanyyy/chest-ctscan-images/versions/1/Data

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

try:
    # Define paths for train, valid, and test sets
    train_dir = os.path.join(base_dir, 'train')
    valid_dir = os.path.join(base_dir, 'valid')
    test_dir = os.path.join(base_dir, 'test')

    # Create datasets using ImageFolder for each split
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    val_dataset = datasets.ImageFolder(valid_dir, transform=test_transform) # Use test_transform for validation
    test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

    # Determine the number of classes from the training dataset
    classes = train_dataset.classes
    num_classes = len(classes)

    # Create DataLoaders
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=False) # No shuffling for validation
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=4, shuffle=False)

    print(f"[INFO] Found {len(train_dataset)} training images, {len(val_dataset)} validation images, and {len(test_dataset)} testing images.")
    print(f"[INFO] Classes: {classes}")
except Exception as e:
    print(f"[ERROR] Data loading failed. Check dataset path and structure: {e}")

[INFO] Found 613 training images, 72 validation images, and 315 testing images.
[INFO] Classes: ['adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib', 'large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa', 'normal', 'squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa']


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Improved Novel Hybrid Multi-Model Architecture

This section defines the significantly enhanced hybrid ensemble. It leverages pre-trained weights for ViT, DenseNet, and EfficientNet, and introduces a specialized, robust `ImprovedCustomCNN` for micro-nodule detection. A novel adaptive fusion module is integrated to intelligently combine features from all branches.

In [5]:
import torch.nn as nn # Added import
import timm # Added import for robustness
import torchvision.models as models # Added import for robustness
import torch.nn.functional as F # Added import for robustness

class ImprovedCustomCNN(nn.Module):
    """
    Improved Custom CNN for micro-nodule detection.
    Key innovations:
    1. Only 2 MaxPool layers (preserves small nodule details)
    2. Residual connections for better gradient flow
    3. Batch normalization after each conv
    4. Squeeze-and-Excitation blocks for channel attention
    """
    def __init__(self, in_channels=3, out_features=128):
        super(ImprovedCustomCNN, self).__init__()

        # Stage 1: Preserve resolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.pool1 = nn.MaxPool2d(2, stride=2)  # 224 → 112

        # Stage 2: Extract mid-level features
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool2 = nn.MaxPool2d(2, stride=2)  # 112 → 56

        # Stage 3: High-level features (no more pooling!)
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        # Stage 4: Squeeze-and-Excitation block (channel attention)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 128),
            nn.Sigmoid()
        )

        # Global feature aggregation
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Store input for residual connection
        identity = x

        # Stage 1
        out = self.conv1(x)
        out = self.pool1(out)

        # Stage 2
        out = self.conv2(out)
        out = self.pool2(out)

        # Stage 3
        out = self.conv3(out)

        # Squeeze-and-Excitation
        se_weights = self.se(out)
        out = out * se_weights.unsqueeze(-1).unsqueeze(-1)

        # Global pooling and output
        out = self.global_pool(out)
        out = out.flatten(1)
        out = self.fc(out)

        return out


class NovelHybridEnsembleImproved(nn.Module):
    """
    Improved Novel Hybrid Ensemble for Lung Cancer Classification

    Architecture:
    - Branch 1: Vision Transformer (pre-trained) → Global context
    - Branch 2: DenseNet121 (pre-trained) → Deep hierarchical features
    - Branch 3: EfficientNet-B0 (pre-trained) → Efficient balanced features
    - Branch 4: Improved Custom CNN (scratch) → Micro-nodule specialization

    Novel aspects:
    1. Four-branch parallel fusion
    2. Custom CNN with SE blocks and residual connections
    3. Adaptive feature fusion
    """
    def __init__(self, num_classes=2, use_pretrained=True):
        super(NovelHybridEnsembleImproved, self).__init__()

        # Branch 1: Vision Transformer (Global Context Expert)
        self.vit = timm.create_model(
            'vit_large_patch16_224',
            pretrained=use_pretrained,
            num_classes=0  # Remove classification head, keep features
        )
        vit_features = self.vit.num_features  # 1024 for vit_large

        # Branch 2: DenseNet121 (Deep Feature Expert)
        # Updated: Use `weights` parameter instead of `pretrained` for torchvision models
        densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if use_pretrained else None)
        self.densenet_features = densenet.features
        densenet_features_out = 1024  # DenseNet121's final channels

        # Branch 3: EfficientNet-B0 (Efficiency Expert)
        self.efficientnet = timm.create_model(
            'efficientnet_b0',
            pretrained=use_pretrained,
            num_classes=0
        )
        efficientnet_features = self.efficientnet.num_features  # 1280

        # Branch 4: Improved Custom CNN (Micro-Nodule Specialist)
        self.custom_cnn = ImprovedCustomCNN(in_channels=3, out_features=256)
        custom_features = 256

        # Total combined features
        total_features = vit_features + densenet_features_out + efficientnet_features + custom_features
        print(f"[INFO] Total feature dimension: {total_features}")

        # Adaptive Fusion Module (instead of simple concatenation)
        self.fusion_gate = nn.Sequential(
            nn.Linear(total_features, total_features // 4),
            nn.ReLU(),
            nn.Linear(total_features // 4, total_features),
            nn.Sigmoid()
        )

        # Final Classifier with progressive dimension reduction
        self.classifier = nn.Sequential(
            nn.Linear(total_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Extract features from each branch
        vit_feats = self.vit(x)                    # Global context
        densenet_feats = self._forward_densenet(x)  # Deep features
        efficient_feats = self.efficientnet(x)      # Efficient features
        custom_feats = self.custom_cnn(x)           # Micro-nodule features

        # Concatenate all features
        combined = torch.cat([vit_feats, densenet_feats, efficient_feats, custom_feats], dim=1)

        # Adaptive gating (novel fusion mechanism)
        gate_weights = self.fusion_gate(combined)
        gated_features = combined * gate_weights

        # Classification
        output = self.classifier(gated_features)
        return output

    def _forward_densenet(self, x):
        """Helper function for DenseNet feature extraction"""
        features = self.densenet_features(x)
        features = F.relu(features, inplace=True)
        features = F.adaptive_avg_pool2d(features, (1, 1))
        features = torch.flatten(features, 1)
        return features


# =========================
# INSTANTIATE MODEL
# =========================
def create_model(num_classes=2, use_pretrained=True):
    """
    Factory function to create the improved hybrid model.

    Args:
        num_classes: Number of output classes (2 for benign/malignant)
        use_pretrained: Use ImageNet pre-trained weights for ViT, DenseNet, EfficientNet

    Returns:
        model: ImprovedNovelHybridEnsemble instance
    """
    model = NovelHybridEnsembleImproved(
        num_classes=num_classes,
        use_pretrained=use_pretrained
    )

    # Count total parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"[INFO] Model created successfully!")
    print(f"[INFO] Total parameters: {total_params:,}")
    print(f"[INFO] Trainable parameters: {trainable_params:,}")

    return model

try:
    # num_classes is determined from the data loader setup
    model = create_model(num_classes=num_classes, use_pretrained=True).to(device)
    print(f"[INFO] NovelHybridEnsembleImproved created for {num_classes} classes (using pre-trained weights).")
except NameError:
    print("[ERROR] 'classes' not defined. Please run previous data loading cells first.")
except Exception as e:
    print(f"[ERROR] Model creation failed: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[INFO] Total feature dimension: 3584
[INFO] Model created successfully!
[INFO] Total parameters: 323,021,888
[INFO] Trainable parameters: 323,021,888
[INFO] NovelHybridEnsembleImproved created for 4 classes (using pre-trained weights).


### Visualize Model Attention with Grad-CAM

To understand *why* the model makes certain predictions, especially for identifying 'damage' or key features, we can use techniques like Grad-CAM (Gradient-weighted Class Activation Mapping). Grad-CAM generates a heatmap that highlights the regions in the input image that were most important for activating the final classification layer.

In [6]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks to capture gradients and activations
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        output = self.model(x) # Perform forward pass

        if class_idx is None:
            # Get the index of the highest scoring class if not provided
            class_idx = output.argmax(dim=1)

        # Zero out all but the selected class to backpropagate specific class gradients
        one_hot = torch.zeros_like(output)
        one_hot[0][class_idx] = 1

        output.backward(gradient=one_hot, retain_graph=True) # Perform backward pass

        # Retrieve stored gradients and activations
        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]

        # Global average pooling of gradients to get channel weights
        weights = np.mean(gradients, axis=(1, 2))

        # Create heatmap: weighted sum of activations
        heatmap = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            heatmap += w * activations[i]

        heatmap = np.maximum(heatmap, 0) # Apply ReLU
        heatmap /= np.max(heatmap) if np.max(heatmap) > 0 else 1e-10 # Normalize to [0, 1]

        return heatmap

def visualize_gradcam(original_image_pil, heatmap, title="Grad-CAM Visualization", alpha=0.5):
    # Resize heatmap to original image dimensions
    heatmap = cv2.resize(heatmap, (original_image_pil.width, original_image_pil.height))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET) # Apply JET colormap

    # Convert PIL image to OpenCV format (BGR for OpenCV, then RGB for matplotlib)
    img_np = np.array(original_image_pil)
    img_cv = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    # Overlay heatmap on original image
    superimposed_img = heatmap * alpha + img_cv * (1 - alpha)
    superimposed_img = np.uint8(superimposed_img)
    superimposed_img = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB) # Convert back to RGB

    plt.figure(figsize=(8, 8))
    plt.imshow(superimposed_img)
    plt.title(title)
    plt.axis('off')
    plt.show()


### Improved Training Strategy

This section defines a sophisticated training loop, `train_improved_model`, incorporating best practices for deep learning: different learning rates for various model components, a cosine annealing learning rate scheduler, gradient clipping to prevent exploding gradients, and early stopping to prevent overfitting on the validation set. This strategy aims for more stable and effective training, crucial for complex hybrid models.

In [ ]:
# =========================
# IMPROVED TRAINING LOOP
# =========================
def train_improved_model(model, train_loader, val_loader, epochs=100):
    """
    Improved training with:
    1. Different learning rates for different branches
    2. Gradient clipping
    3. Early stopping
    4. Learning rate warmup (implicitly handled by CosineAnnealingLR if set up carefully)
    """

    # Different learning rates for different components
    # Custom CNN trains faster (from scratch), pre-trained models fine-tune slower
    optimizer = optim.AdamW([
        {'params': model.vit.parameters(), 'lr': 1e-5},
        {'params': model.densenet_features.parameters(), 'lr': 1e-5},
        {'params': model.efficientnet.parameters(), 'lr': 1e-5},
        {'params': model.custom_cnn.parameters(), 'lr': 1e-4},  # Higher LR for scratch training
        {'params': model.fusion_gate.parameters(), 'lr': 1e-3},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ], weight_decay=1e-4)

    # Cosine annealing scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0
    early_stop_counter = 0

    print("\n[INFO] Starting Improved Training Loop (with pre-trained weights and advanced strategies). This will take time...")

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            if (batch_idx + 1) % 50 == 0: # Print loss every 50 batches
                print(f"  Batch [{batch_idx+1}/{len(train_loader)}] - Loss: {loss.item():.4f}")

        avg_train_loss = train_loss/len(train_loader)

        # Validation phase
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step()

        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}")

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            early_stop_counter = 0
            torch.save(model.state_dict(), 'best_model.pth') # Save the best model
            print("  [INFO] Saved best model with validation accuracy: {:.4f}".format(best_val_acc))
        else:
            early_stop_counter += 1
            if early_stop_counter >= 10:
                print(f"  [INFO] Early stopping at epoch {epoch+1} (no improvement for 10 epochs).")
                break

    print("\n[INFO] Training finished.")
    return model

if 'train_loader' in locals() and 'val_loader' in locals() and 'model' in locals():
    model = train_improved_model(model, train_loader, val_loader, epochs=100) # Increased epochs for better learning
else:
    print("[ERROR] 'train_loader', 'val_loader', or 'model' not defined. Please run previous cells.")


[INFO] Starting Improved Training Loop (with pre-trained weights and advanced strategies). This will take time...


/tmp/ipykernel_8386/3331238844.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/tmp/ipykernel_8386/3331238844.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
/tmp/ipykernel_8386/3331238844.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch

### Model Evaluation and Advanced Plots

After training, it's crucial to evaluate the model's performance on unseen data. This section loads the best-performing model (saved during training) and calculates various metrics like accuracy, precision, recall, and F1-score. It also generates a confusion matrix to visualize the model's classification performance across different classes.

In [ ]:
# =========================
# MODEL EVALUATION
# =========================
def evaluate_model(model, test_loader, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Convert to numpy arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    print("\n[INFO] Model Evaluation Results:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

    # Classification Report
    print("\n[INFO] Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=classes, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

# Load the best model state dictionary
try:
    model.load_state_dict(torch.load('best_model.pth'))
    print("[INFO] Best model loaded successfully for evaluation.")
    evaluate_model(model, test_loader, classes)
except FileNotFoundError:
    print("[ERROR] 'best_model.pth' not found. Please ensure training completed successfully.")
except Exception as e:
    print(f"[ERROR] Error during model evaluation: {e}")

### Test with Your Own Image

This section provides a utility to upload a new image and get a prediction from the trained model. It also includes a confidence check: if the model's highest prediction confidence for any class is below a certain threshold (defaulting to 60%), it will suggest that the image might not be a valid lung scan or is of poor quality. This helps filter out irrelevant images before prediction.

In [ ]:
# =========================
# PREDICT ON CUSTOM IMAGE
# =========================
def predict_uploaded_image(image_bytes, model, transform, classes, confidence_threshold=0.6):
    model.eval()
    try:
        # Open the image from bytes
        image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        original_image_pil = image.copy() # Keep a copy of the original PIL image for Grad-CAM visualization

        # Apply the same transformations as test set
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = F.softmax(outputs, dim=1)

            # Get highest confidence prediction
            max_prob, predicted_idx = torch.max(probabilities, 1)
            max_prob_value = max_prob.item()
            predicted_class = classes[predicted_idx.item()]

            if max_prob_value < confidence_threshold:
                print(f"[WARNING] Low prediction confidence ({max_prob_value:.2f}). This image might not be a valid lung scan or could be of poor quality. Please upload a clear lung CT scan.")
                return None, None, None, None, None # Indicate ambiguous image and return None for extra values
            else:
                print(f"[INFO] Predicted Class: {predicted_class} (Confidence: {max_prob_value:.2f})")
                return predicted_class, max_prob_value, original_image_pil, input_tensor, predicted_idx.item()

    except Exception as e:
        print(f"[ERROR] Could not process image: {e}")
        return None, None, None, None, None

# Function to handle file upload
def upload_and_predict():
    print("[INFO] Please upload a lung CT scan image.")
    uploaded = files.upload()

    if uploaded:
        for fn in uploaded.keys():
            print(f'[INFO] User uploaded file "{fn}"')
            # Retrieve image_bytes from the uploaded dictionary for the specific file
            image_bytes = uploaded[fn]
            predicted_class, confidence, original_image, input_tensor, predicted_idx = predict_uploaded_image(image_bytes, model, test_transform, classes)

            if predicted_class:
                print(f"[RESULT] The model predicts this is a '{predicted_class}' with {confidence*100:.2f}% confidence.")

                # Initialize Grad-CAM targeting the last convolutional layer of the Custom CNN
                # model.custom_cnn.conv3[5] is the BatchNorm2d layer before ReLU in the last conv block
                gradcam = GradCAM(model, model.custom_cnn.conv3[5])
                heatmap = gradcam(input_tensor, class_idx=predicted_idx)

                # Visualize Grad-CAM
                visualize_gradcam(original_image, heatmap, title=f"Grad-CAM for: {predicted_class} (Conf: {confidence*100:.2f}%), Image: {fn}")

            else:
                print("[RESULT] Model could not confidently classify the image. Please try another image.")
    else:
        print("[INFO] No file uploaded.")

# --- Test Case: Upload and Predict ---
print("\n--- Running Test Case: Upload your own image ---")
# You will be prompted to upload an image file from your local machine.
# Choose a lung CT scan image to test the model.

upload_and_predict()